# Assignment 10 - Problem 1
## Turning an English request into SQL with the OpenAI API

**Name:** Hussein Jawad

The request I need to convert to SQL:

> "Select all customers who are born in either June, July, October, live in Boston, are senior citizens and hence qualify for Medicare benefit"

Contents:
- OpenAI account and API key
- Part a - Install the OpenAI library
- Part b - Ask OpenAI to write the SQL
- Part c - Choose the prompt and test temperature and other settings for accuracy
- Part d - The JSON response
- Appendix - Concepts used in this problem

## OpenAI account and API key

I created an account on platform.openai.com, added billing credits, and created a new secret key under **API keys**. I named the key `e89-hw10-key`. The key is stored in a `.env` file in the same folder as this notebook, not in the notebook itself:

```
OPENAI_API_KEY=sk-...
OPENAI_KEY_NAME=e89-hw10-key
```

## Part a - Install the OpenAI library and set up the client

In [1]:
# Part a: install the OpenAI Python library and python-dotenv (reads the .env file)
%pip install -q --upgrade openai python-dotenv

I run the notebook in Google Colab, and the notebook and `.env` file are in my Google Drive folder. Colab starts in `/content`, so I mount Drive and change into that folder first. If the notebook runs on my own computer instead, the Drive part is skipped.

In [3]:
import os   # file and folder functions

PROJECT_DIR = "/content/drive/MyDrive/CSCI-E-89-1-Deep-Learning/HW10"   # Drive folder with this notebook and .env

try:
    from google.colab import drive   # this import only works inside Colab
    drive.mount("/content/drive")    # connect my Google Drive to the Colab machine
    os.chdir(PROJECT_DIR)            # work inside the project folder so .env is found
except ImportError:
    pass                             # not in Colab: stay in the current folder

print("Working folder:", os.getcwd())               # show the folder we are working in
print(".env found    :", os.path.exists(".env"))    # check that the key file is there

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working folder: /content/drive/MyDrive/CSCI-E-89-1-Deep-Learning/HW10
.env found    : True


Load the key from `.env` and create the client. If OpenAI returns `model_not_found`, change `MODEL` / `SMALL_MODEL`.

In [4]:
import json                                     # convert between JSON text and Python dictionaries
import openai                                   # the OpenAI library (used here to print its version)
from dotenv import load_dotenv, find_dotenv     # read variables from the .env file
from openai import OpenAI                       # the client class used for all API calls

load_dotenv(find_dotenv(usecwd=True), override=True)   # load OPENAI_API_KEY and OPENAI_KEY_NAME from .env

MODEL = "gpt-5.4"               # main model
SMALL_MODEL = "gpt-5.4-mini"    # smaller, cheaper model used for comparison in part c
client = OpenAI()               # create the client; it reads OPENAI_API_KEY from the environment

key = os.getenv("OPENAI_API_KEY")                                 # get the key that was loaded
assert key, "OPENAI_API_KEY not found - check the .env file"      # stop here if the key is missing
print("openai version:", openai.__version__)                      # library version
print("Key name      :", os.getenv("OPENAI_KEY_NAME"))            # the name I gave the key
print("Key           :", key[:3] + "..." + key[-4:])              # masked key, never the full key

def show_json(obj):
    # print an API response object as a readable dictionary (helper from the lab notebooks)
    display(json.loads(obj.model_dump_json()))

openai version: 3.52.0
Key name      : e89-hw10-key
Key           : sk-...iIoA


## Part b - Ask OpenAI to write the SQL

The assignment doesn't give a table, so I made a `Customers` table. It needs a `DateOfBirth` column to get the birth month and the age. I also added an `Orders` table that the query should not need. Being a senior citizen who qualifies for Medicare means being 65 or older.

In [5]:
# Table definitions given to the model. The assignment doesn't give a schema, so I made one:
# DateOfBirth gives the birth month and the age; the Orders table is not needed for this request.
SCHEMA = '''
CREATE TABLE Customers (
  CustomerID   INTEGER PRIMARY KEY,
  FirstName    VARCHAR(50),
  LastName     VARCHAR(50),
  DateOfBirth  DATE,          -- 'YYYY-MM-DD'
  Email        VARCHAR(100),
  Phone        VARCHAR(20),
  City         VARCHAR(50),
  State        CHAR(2)        -- e.g. 'MA'
);

CREATE TABLE Orders (
  OrderID      INTEGER PRIMARY KEY,
  CustomerID   INTEGER REFERENCES Customers(CustomerID),
  OrderDate    DATE,
  TotalAmount  DECIMAL(10, 2)
);
'''.strip()   # remove the empty first and last lines

# The English request from the assignment, word for word
REQUEST = ("Select all customers who are born in either June, July, October, live in Boston, "
           "are senior citizens and hence qualify for Medicare benefit")

First I used the same kind of prompt as the SQL example in the lab: the tables go in the system message and the request in the user message.

In [6]:
# First prompt, same pattern as the SQL example in the lab:
# the tables go in the system message and the request in the user message
baseline_messages = [
    {"role": "system",                                   # system message: instructions + tables
     "content": "Given the following SQL tables, your job is to write queries given a user's request.\n\n" + SCHEMA},
    {"role": "user", "content": REQUEST},                # user message: the English request
]

baseline_response = client.responses.create(   # send the request to OpenAI
    model=MODEL,                                # gpt-5.4
    input=baseline_messages,                    # the two messages above
    reasoning={"effort": "none"},               # no reasoning, so temperature can be set
    temperature=0,                              # least random, most consistent output
    max_output_tokens=1000,                     # upper limit on the answer length
)
print(baseline_response.output_text)            # the model's answer as plain text

```sql
SELECT *
FROM Customers
WHERE City = 'Boston'
  AND EXTRACT(MONTH FROM DateOfBirth) IN (6, 7, 10)
  AND DateOfBirth <= CURRENT_DATE - INTERVAL '65 years';
```


The model wrote reasonable SQL, but there are problems:
- It's plain text with the SQL inside a code block, so a program can't use it directly.
- The prompt doesn't say which database it is for, so the model used PostgreSQL syntax (`EXTRACT(MONTH FROM ...)`, `CURRENT_DATE - INTERVAL '65 years'`).
- It only checks the city, not the state.

So in part c I improved the prompt and asked for the answer as JSON.

## Part c - Choosing the prompt and testing settings for accuracy

### c.1 A better prompt

What I changed compared to the first prompt:
- The request goes between `<request>` tags and the tables between `<schema>` tags (delimiters from Notebook 1).
- I ask for SQLite, so I can run the query inside this notebook, and tell it not to hard-code today's date.
- It has to list the conditions it found before it writes the SQL, so it works the problem out first.
- The answer has to be JSON. Instead of only asking for JSON in the prompt, I used Structured Outputs (`text.format` with a JSON schema), so the reply always parses.

In [7]:
# Better prompt (system message): the tables inside <schema> tags plus clear rules.
# The rules fix the problems of the first prompt: they name the database (SQLite), say how dates
# are stored, define "Boston", ask for the exact age, and ask for the conditions before the SQL.
SYSTEM_PROMPT = f'''
You are an expert SQL developer. Translate the request delimited by <request></request>
into one SQL query for this database:

<schema>
{SCHEMA}
</schema>

Rules:
- Use SQLite syntax and only the tables and columns above.
- Write a single read-only SELECT statement.
- Dates are text 'YYYY-MM-DD'. Use date('now') for today, never a hard-coded date.
- If a condition depends on age, compute the exact age.
- "Boston" means City = 'Boston' and State = 'MA'.
- "Select all customers" means return all columns.
- First list the conditions you found in the request, then write the query.
- If the request can't be answered with these tables, set "sql" to null and explain why.
'''.strip()   # remove the empty first and last lines

# JSON schema that the answer must follow (Structured Outputs)
SQL_JSON_SCHEMA = {
    "type": "object",                                                    # the answer is one JSON object
    "properties": {
        "conditions":  {"type": "array", "items": {"type": "string"}},   # conditions found in the request
        "assumptions": {"type": "array", "items": {"type": "string"}},   # e.g. Medicare means age 65+
        "sql":         {"type": ["string", "null"]},                     # the query, or null if not possible
        "explanation": {"type": "string"},                               # how the query works
    },
    "required": ["conditions", "assumptions", "sql", "explanation"],     # all four fields must be there
    "additionalProperties": False,                                       # no other fields allowed
}
# Tell the API to force the answer into this schema (strict = always matches the schema)
TEXT_FORMAT = {"format": {"type": "json_schema", "name": "nl_to_sql",
                          "schema": SQL_JSON_SCHEMA, "strict": True}}

I use one function for all calls so I can change the prompt, the model and the settings easily. Following Notebook 1, temperature and top_p are only allowed when reasoning effort is `"none"`, so the function leaves them out for other effort levels.

In [8]:
def generate_sql(prompt="engineered", model=MODEL, temperature=None, top_p=None, effort="none"):
    # Send one request and return the full response object.
    # prompt is "baseline" (first prompt) or "engineered" (better prompt with a JSON answer).
    if prompt == "baseline":
        args = {"input": baseline_messages}                                       # first prompt, plain-text answer
    else:
        args = {"input": [{"role": "system", "content": SYSTEM_PROMPT},           # rules + tables
                          {"role": "user", "content": f"<request>{REQUEST}</request>"}],   # request inside tags
                "text": TEXT_FORMAT}                                              # force the JSON schema
    if effort == "none":                                       # sampling settings are only allowed without reasoning
        if temperature is not None: args["temperature"] = temperature              # how random the output is
        if top_p is not None:       args["top_p"] = top_p                          # keep only the most likely tokens
    max_tokens = 1000 if effort == "none" else 4000            # reasoning tokens count too, so allow more
    return client.responses.create(model=model, reasoning={"effort": effort},      # make the API call
                                   max_output_tokens=max_tokens, **args)

response = generate_sql("engineered", MODEL, temperature=0)   # better prompt, gpt-5.4, temperature 0
result = json.loads(response.output_text)                     # the answer is JSON text -> Python dictionary
print("status:", response.status)                             # completed = the answer was not cut off
print(json.dumps(result, indent=2))                           # show the whole JSON answer
print("\nSQL:\n" + result["sql"])                             # show the SQL on its own

status: completed
{
  "conditions": [
    "Return all columns from Customers",
    "Customer lives in Boston (City = 'Boston' and State = 'MA')",
    "Customer was born in June, July, or October",
    "Customer is a senior citizen and qualifies for Medicare benefit, interpreted as age 65 or older",
    "Age must be computed exactly using today's date"
  ],
  "assumptions": [
    "\"Senior citizens and hence qualify for Medicare benefit\" is interpreted as customers who are at least 65 years old",
    "Month of birth is determined from DateOfBirth using SQLite strftime('%m', DateOfBirth)"
  ],
  "sql": "SELECT *\nFROM Customers\nWHERE City = 'Boston'\n  AND State = 'MA'\n  AND strftime('%m', DateOfBirth) IN ('06', '07', '10')\n  AND (\n    CAST(strftime('%Y', date('now')) AS INTEGER)\n    - CAST(strftime('%Y', DateOfBirth) AS INTEGER)\n    - (strftime('%m-%d', date('now')) < strftime('%m-%d', DateOfBirth))\n  ) >= 65;",
  "explanation": "The query returns all customer columns for people

This time the model listed the conditions first and worked out on its own that "qualify for Medicare" means age 65 or older. It computes the exact age: this year minus the birth year, minus 1 more if the birthday hasn't happened yet this year.

### c.2 How I measure accuracy

A query can look right and still be wrong, so I tested it. I made a small SQLite database where each customer tests one condition (wrong month, wrong city, too young, exactly 65, missing birth date). I worked out the correct answer in Python and compared it with what the query returns. The birth dates are based on today's date, so the test still works if I run it later.

In [9]:
import re, sqlite3                                  # regular expressions and the SQLite database
import pandas as pd                                 # tables for the results
from datetime import date, datetime, timezone       # dates for the test data

TODAY = datetime.now(timezone.utc).date()   # today's date in UTC, the same as SQLite's date('now')

def born(years_ago, month, day):
    # birth date `years_ago` years before this year, e.g. born(72, 6, 15) = June 15, 72 years ago
    return date(TODAY.year - years_ago, month, day).isoformat()

# Test customers: (id, first, last, date of birth, city, state)   and what each row tests
CUSTOMERS = [
    (1,  "Mary",  "Smith",  born(72, 6, 15),  "Boston",    "MA"),   # match (June)
    (2,  "John",  "Brown",  born(80, 7, 4),   "Boston",    "MA"),   # match (July)
    (3,  "Ann",   "Lee",    born(66, 10, 31), "Boston",    "MA"),   # match (October)
    (4,  "Eve",   "Clark",  born(65, 10, 1),  "Boston",    "MA"),   # exactly 65 (>= vs >)
    (5,  "Lena",  "Wong",   born(85, 6, 30),  "Boston",    "MA"),   # match
    (6,  "Bob",   "Jones",  born(75, 5, 10),  "Boston",    "MA"),   # wrong month
    (7,  "Grace", "Hill",   born(90, 12, 1),  "Boston",    "MA"),   # wrong month
    (8,  "Carl",  "Davis",  born(70, 6, 1),   "Cambridge", "MA"),   # wrong city
    (9,  "Dana",  "White",  born(40, 6, 20),  "Boston",    "MA"),   # too young
    (10, "Frank", "Green",  born(64, 7, 15),  "Boston",    "MA"),   # 63 or 64, too young
    (11, "Omar",  "King",   None,             "Boston",    "MA"),   # no birth date
]

def build_test_db():
    # Create a new in-memory SQLite database that holds the test customers
    con = sqlite3.connect(":memory:")                                       # database that only lives in memory
    con.executescript(SCHEMA)                                               # same tables as in the prompt
    con.executemany("INSERT INTO Customers VALUES (?, ?, ?, ?, ?, ?, ?, ?)",               # add the customers
                    [(i, f, l, dob, f.lower() + "@example.com", "617-555-01%02d" % i, city, st)   # + email, phone
                     for i, f, l, dob, city, st in CUSTOMERS])
    con.execute("INSERT INTO Orders VALUES (1, 1, '2026-01-05', 120.50)")  # one order (the query doesn't need it)
    con.execute("PRAGMA query_only = ON")                                   # read-only: generated SQL can't change data
    return con

def age(dob):
    # exact age today: difference in years, minus 1 if this year's birthday hasn't happened yet
    d = date.fromisoformat(dob)                                                  # text date -> date
    return TODAY.year - d.year - ((TODAY.month, TODAY.day) < (d.month, d.day))   # True counts as 1

# Correct answer, worked out in Python (not with SQL)
EXPECTED = {i for i, f, l, dob, city, st in CUSTOMERS
            if dob and date.fromisoformat(dob).month in (6, 7, 10)    # born in June, July or October
            and city == "Boston" and st == "MA" and age(dob) >= 65}   # lives in Boston, MA and is 65 or older
print("Today:", TODAY, "- expected CustomerIDs:", sorted(EXPECTED))   # show the expected answer

Today: 2026-11-17 - expected CustomerIDs: [1, 2, 3, 4, 5]


`check_sql` runs a query on the test database and compares the returned IDs with the expected ones. For the first prompt it takes the SQL out of the code block, and it refuses anything that isn't a SELECT.

In [10]:
def extract_sql(text):
    # Get the SQL out of a plain-text answer: the part inside the ```sql code block, or the whole text
    m = re.search(r"```(?:sql|sqlite)?\s*(.*?)```", text, re.S | re.I)   # look for a code block
    return (m.group(1) if m else text).strip()                          # its content without extra spaces

def check_sql(sql):
    # Run a query on the test database and compare the customers it returns with EXPECTED
    out = {"runs": False, "correct": False, "ids": None, "error": None}   # result if something goes wrong
    if not sql or not re.match(r"^\s*(WITH|SELECT)\b", sql, re.I):          # only accept read-only queries
        out["error"] = "no SELECT statement"
        return out
    try:
        cur = build_test_db().execute(sql)                                  # run the query on a fresh test database
        cols = [c[0].lower() for c in cur.description]                      # column names of the result
        ids = {row[cols.index("customerid")] for row in cur.fetchall()}     # CustomerIDs that were returned
        out.update(runs=True, ids=sorted(ids), correct=(ids == EXPECTED))   # correct = exactly the expected customers
    except Exception as e:                                                  # e.g. syntax from another SQL dialect
        out["error"] = f"{type(e).__name__}: {e}"                           # keep the error message
    return out

baseline_sql = extract_sql(baseline_response.output_text)   # SQL from the first prompt's text answer
print("First prompt :", check_sql(baseline_sql))           # test the first prompt's query
print("Better prompt:", check_sql(result["sql"]))           # test the better prompt's query

First prompt : {'runs': False, 'correct': False, 'ids': None, 'error': 'OperationalError: near "FROM": syntax error'}
Better prompt: {'runs': True, 'correct': True, 'ids': [1, 2, 3, 4, 5], 'error': None}


### c.3 Trying different settings

From Notebook 1: temperature controls how random the output is (0 = almost always the same), top_p limits the choice to the most likely tokens, and reasoning effort lets the model think first (temperature and top_p can't be used then). I also compared the smaller model. I ran every setting 3 times, checked each query with `check_sql`, and counted how many different queries each setting gave.

In [11]:
import time   # to measure how long each call takes

# Settings to compare: (label, prompt, model, settings passed to generate_sql)
SETTINGS = [
    ("first prompt, T=0",      "baseline",   MODEL,       {"temperature": 0}),     # lab-style prompt
    ("better prompt, T=0",     "engineered", MODEL,       {"temperature": 0}),     # no randomness
    ("better prompt, T=0.7",   "engineered", MODEL,       {"temperature": 0.7}),   # some randomness
    ("better prompt, T=1.5",   "engineered", MODEL,       {"temperature": 1.5}),   # a lot of randomness
    ("better prompt, top_p=0.1", "engineered", MODEL,     {"top_p": 0.1}),         # only the most likely tokens
    ("better prompt, mini, T=0", "engineered", SMALL_MODEL, {"temperature": 0}),   # smaller model
    ("better prompt, effort=low", "engineered", MODEL,    {"effort": "low"}),      # reasoning instead of sampling
]
N_RUNS = 3   # repeat each setting 3 times to see how consistent it is

rows = []                                                   # one row per API call
for label, prompt, model, settings in SETTINGS:             # for each setting...
    for run in range(N_RUNS):                               # ...run it 3 times
        start = time.time()                                 # start the timer
        try:
            r = generate_sql(prompt, model, **settings)     # call the API
            sql = extract_sql(r.output_text) if prompt == "baseline" else json.loads(r.output_text)["sql"]   # get the SQL
            rows.append({"setting": label, "sql": sql, "tokens": r.usage.total_tokens,     # save the SQL, tokens,
                         "seconds": time.time() - start, **check_sql(sql)})                # time and accuracy check
        except Exception as e:   # e.g. a setting the model doesn't support
            rows.append({"setting": label, "sql": None, "runs": False, "correct": False,   # record the error
                         "error": f"{type(e).__name__}: {e}"[:150]})                      # instead of stopping
    print("done:", label)                                   # progress message

runs = pd.DataFrame(rows)                                   # all calls in one table
summary = runs.groupby("setting", sort=False).agg(          # one row per setting:
    accuracy=("correct", "mean"),                           # share of runs that returned exactly the right customers
    runs_on_sqlite=("runs", "mean"),                        # share of runs whose SQL ran without an error
    different_queries=("sql", lambda s: s.dropna().str.strip().nunique()),   # number of different SQL texts
    avg_tokens=("tokens", "mean"),                          # average tokens per call
    avg_seconds=("seconds", "mean"),                        # average time per call in seconds
).round(2)                                                  # round to 2 decimals
summary                                                     # show the table

done: first prompt, T=0
done: better prompt, T=0
done: better prompt, T=0.7
done: better prompt, T=1.5
done: better prompt, top_p=0.1
done: better prompt, mini, T=0
done: better prompt, effort=low


,accuracy,runs_on_sqlite,different_queries,avg_tokens,avg_seconds
setting,,,,,
"first prompt, T=0",0.0,0.0,2,239.67,2.69
"better prompt, T=0",1.0,1.0,1,660.00,5.26
"better prompt, T=0.7",1.0,1.0,2,676.00,6.41
"better prompt, T=1.5",1.0,1.0,3,668.33,7.89
"better prompt, top_p=0.1",1.0,1.0,1,660.00,5.24
"better prompt, mini, T=0",1.0,1.0,2,643.67,3.13
"better prompt, effort=low",1.0,1.0,2,738.67,6.13


### Observations

- **The prompt mattered the most.** The first prompt got 0 out of 3 right. Its logic was close (it only missed the state check), but it was written for PostgreSQL (`EXTRACT(MONTH FROM DateOfBirth)`, `INTERVAL '65 years'`), so SQLite gave `near "FROM": syntax error` every time. The better prompt names the database and the date format, and it was correct in all 18 of its runs, at every setting.
- **JSON.** With Structured Outputs, every run of the better prompt returned valid JSON, even at temperature 1.5.
- **Temperature.** Higher temperature gave more variety: the same query in all 3 runs at T=0, 2 different queries at T=0.7 and 3 different queries at T=1.5. All of them were still correct, so only the way the query was written changed, not the result. Temperature 0 wasn't perfectly repeatable either: the first prompt and the mini model each gave 2 different queries at T=0, and the better prompt wrote the month check two ways in c.1 and part d (`strftime('%m', DateOfBirth) IN ('06', '07', '10')` and `CAST(strftime('%m', DateOfBirth) AS INTEGER) IN (6, 7, 10)`). Both are correct.
- **top_p = 0.1** acted like temperature 0 and gave the same query in all 3 runs.
- **Reasoning effort "low"** used more tokens (739 vs 660 on average) and took longer (6.1 s vs 5.3 s) with no gain in accuracy, because the task is simple once the prompt is clear.
- **The smaller model** (gpt-5.4-mini) was also 100% correct and the fastest (3.1 s vs 5.3 s), but less consistent.
- **Cost.** The better prompt uses more tokens than the first one (about 660 vs 240) because of the longer instructions and the JSON fields, but that is still a very small request.

For part d I used the better prompt with `gpt-5.4`, temperature 0 and no reasoning: it was always correct, the most consistent setting, and used no extra reasoning tokens.

## Part d - The JSON response

The final call uses the settings I picked in part c. `show_json` prints the whole API response, which is itself JSON.

In [13]:
final_response = generate_sql("engineered", MODEL, temperature=0)   # final call with the settings chosen in part c
show_json(final_response)                                           # print the full API response as JSON

The output of the cell above was very long, so I deleted it and copied the important fields here. The model's answer is the text in `output[0].content[0].text` and is shown in full in the next cell. I didn't set `top_p`; 0.98 is the value the API reports by default.

```json
{
  "id": "resp_03443d6f27e97f01006afbe6d8638487d2b2da65a2794a474c",
  "object": "response",
  "model": "gpt-5.4-2026-03-05",
  "status": "completed",
  "error": null,
  "incomplete_details": null,
  "output": [
    {
      "type": "message",
      "role": "assistant",
      "status": "completed",
      "content": [
        {
          "type": "output_text",
          "text": "{\"conditions\":[...],\"assumptions\":[...],\"sql\":\"SELECT * FROM Customers WHERE ...\",\"explanation\":\"...\"}"
        }
      ]
    }
  ],
  "temperature": 0.0,
  "top_p": 0.98,
  "reasoning": {
    "effort": "none"
  },
  "max_output_tokens": 1000,
  "text": {
    "format": {
      "type": "json_schema",
      "name": "nl_to_sql",
      "strict": true
    }
  },
  "usage": {
    "input_tokens": 378,
    "output_tokens": 282,
    "output_tokens_details": {
      "reasoning_tokens": 0
    },
    "total_tokens": 660
  }
}
```

The model's answer parsed from that text, together with the request, the settings, the token usage and my accuracy check:

In [14]:
final_answer = json.loads(final_response.output_text)   # the model's JSON answer -> Python dictionary

# Final JSON: the request, model and settings used, token usage, the model's answer and my accuracy check
final = {
    "name": "Hussein Jawad",                              # my name
    "request": REQUEST,                                   # the English request
    "response_id": final_response.id,                     # id of this API response
    "model": final_response.model,                        # exact model version that answered
    "status": final_response.status,                      # completed = full answer
    "settings": {"temperature": 0, "reasoning_effort": "none", "max_output_tokens": 1000,   # settings used
                 "output_format": "json_schema (strict)"},
    "usage": {"input_tokens": final_response.usage.input_tokens,      # tokens in my prompt
              "output_tokens": final_response.usage.output_tokens,    # tokens in the answer
              "total_tokens": final_response.usage.total_tokens},     # total for this call
    "answer": final_answer,                               # conditions, assumptions, SQL and explanation
    "accuracy_check": check_sql(final_answer["sql"]),     # result of the query on the test database
}
print(json.dumps(final, indent=2))                        # print it as formatted JSON
print("\nFinal SQL:\n" + final_answer["sql"])             # print the SQL on its own

{
  "name": "Hussein Jawad",
  "request": "Select all customers who are born in either June, July, October, live in Boston, are senior citizens and hence qualify for Medicare benefit",
  "response_id": "resp_03443d6f27e97f01006afbe6d8638487d2b2da65a2794a474c",
  "model": "gpt-5.4-2026-03-05",
  "status": "completed",
  "settings": {
    "temperature": 0,
    "reasoning_effort": "none",
    "max_output_tokens": 1000,
    "output_format": "json_schema (strict)"
  },
  "usage": {
    "input_tokens": 378,
    "output_tokens": 282,
    "total_tokens": 660
  },
  "answer": {
    "conditions": [
      "Return all columns from Customers.",
      "Customer lives in Boston (City = 'Boston' and State = 'MA').",
      "Customer was born in June, July, or October.",
      "Customer is a senior citizen, interpreted as age 65 or older as of today.",
      "Use exact age calculation."
    ],
    "assumptions": [
      "\"Qualify for Medicare benefit\" is interpreted as being age 65 or older, since no 

## Appendix - Concepts used in this problem

### 1. The model and the API

**Large language model (LLM).** A neural network (a transformer) trained on a very large amount of text. Given some text, it predicts the next token, adds it to the text and repeats until the answer is complete.

*Here:* `gpt-5.4` is the main model and `gpt-5.4-mini` a smaller, cheaper model.

**Token.** The unit an LLM reads and writes: a word or a piece of a word. Usage and cost are counted in tokens.

*Here:* the `usage` field of a response shows `input_tokens` (my prompt), `output_tokens` (the answer, including any reasoning tokens) and `total_tokens`. `max_output_tokens=1000` limits the length of the answer. The final call used 378 + 282 = 660 tokens.

**API key.** A secret string that identifies my OpenAI account. Every request is billed to it, so anyone who has it can spend my credits.

*Here:* it is stored in a `.env` file, loaded with `load_dotenv()`, and only printed masked (`sk-...iIoA`).

**Responses API.** OpenAI's endpoint for generating text. `client.responses.create(model=..., input=...)` sends a request and returns a response object.

*Here:* I use `output_text` (the answer), `status` (`completed` = finished, `incomplete` = cut off) and `usage` (tokens).

**Message roles.** The input is a list of messages, and each has a role:
- `system`: instructions and context that set the rules for the model
- `user`: the actual request

*Here:* the table definitions and rules are in the system message, and the English request is in the user message.

### 2. Prompting and the output format

**Prompt engineering.** Writing the input so that the model gives the answer you need. The tactics from Notebook 1 that I used:

| Tactic | What I did |
|---|---|
| Use delimiters | Request inside `<request>…</request>`, tables inside `<schema>…</schema>` |
| Ask for a structured output | The answer must be JSON with fixed fields |
| Check conditions | If the tables can't answer the request, return `"sql": null` |
| Work it out first | List the conditions before writing the SQL |

**Structured Outputs (JSON schema).** Asking for JSON only in the prompt usually works but isn't guaranteed. Instead I pass a JSON schema in `text.format` with `strict: True`. The API then guarantees that the answer is valid JSON with exactly these fields: `conditions`, `assumptions`, `sql` and `explanation`. That is why `json.loads(response.output_text)` never failed.

### 3. Settings that change the output

**Temperature.** For every possible next token the model computes a score z. Temperature T turns the scores into probabilities: `p_i = exp(z_i / T) / sum_j exp(z_j / T)`. Example with three tokens A, B, C whose scores are 2, 1 and 0:

| T | A | B | C | Effect |
|---|---|---|---|---|
| 0.5 | 0.87 | 0.12 | 0.02 | A is picked most of the time: more consistent |
| 1.0 | 0.67 | 0.24 | 0.09 | the default |
| 2.0 | 0.51 | 0.31 | 0.19 | flatter: more variety, more risk |

At T = 0 the model picks the most likely token, so the output is nearly the same every run (small differences can still happen). Consistent doesn't mean correct: the model can repeat the same mistake every time.

*Here:* T = 0, 0.7 and 1.5 gave 1, 2 and 3 different queries in 3 runs.

**top_p (nucleus sampling).** Sort the tokens from most to least likely and keep only the top ones until their probabilities add up to p. The model then picks only from those. With the T = 1.0 numbers above, `top_p=0.9` keeps A and B (0.67 + 0.24 = 0.91) and `top_p=0.1` keeps only A. OpenAI recommends changing temperature or top_p, not both.

*Here:* `top_p=0.1` gave the same query in all 3 runs, just like T = 0.

**Reasoning effort.** Reasoning models can "think" with hidden reasoning tokens before they write the answer. `effort` is `none`, `low`, `medium` or `high`: more effort means more tokens and more time, and sometimes better answers on hard problems. With any effort other than `none`, temperature and top_p can't be set.

*Here:* `effort="low"` used more tokens (739 vs 660) and more time with no gain in accuracy, so the final call uses `none`.

### 4. SQL

**Text-to-SQL.** The model translates an English request into SQL. It can only use the correct table and column names if they are in the prompt, which is why the `CREATE TABLE` statements are in the system message.

**SQL dialects.** Every database has its own syntax for some things, especially dates, so the same logic is written differently:

| Task | SQLite | PostgreSQL | MySQL |
|---|---|---|---|
| birth month | `strftime('%m', DateOfBirth)` | `EXTRACT(MONTH FROM DateOfBirth)` | `MONTH(DateOfBirth)` |
| today | `date('now')` | `CURRENT_DATE` | `CURDATE()` |

*Here:* the first prompt didn't name a database, so the model wrote PostgreSQL, which fails on SQLite with `near "FROM": syntax error`.

**Medicare and exact age.** People qualify for Medicare at age 65, so the condition is age >= 65. Exact age = (this year - birth year), minus 1 if this year's birthday hasn't happened yet. Example on 2026-11-17: someone born on 1961-12-20 gives 2026 - 1961 = 65, but their birthday is still to come, so they are 64 and don't qualify. Comparing only the years would get this wrong.

*Here:* the model's SQL subtracts `(strftime('%m-%d', date('now')) < strftime('%m-%d', DateOfBirth))`, which is 1 when the birthday hasn't happened yet and 0 otherwise.

**Testing generated SQL.** An LLM can write SQL that looks right but returns the wrong rows or doesn't run at all.

*Here:* I ran every query on a small test database where I knew the correct answer (worked out in Python). The database was read-only (`PRAGMA query_only = ON`), and a query only counted as correct if it returned exactly customers 1 to 5.